In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
import arff
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

data_path = r'/data/isa/EEG_Eye_State.arff'

with open(data_path, 'r') as f:
    raw = arff.load(f)

columns = [attr[0] for attr in raw['attributes']]
df = pd.DataFrame(raw['data'], columns=columns)

start_time = pd.Timestamp('2025-01-01 00:00:00.000')
interval = pd.to_timedelta('7.8ms')

# Gerando a coluna Timestamp
df['Timestamp'] = [start_time + i * interval for i in range(len(df))]

# Exibindo as primeiras linhas para verificar
print(df.head())

attributes = raw['attributes'] + [('Timestamp', 'STRING')]  # adiciona a nova coluna como string

arff_data = {
    'description': '',
    'relation': 'EEG_Eye_State_with_Timestamp',
    'attributes': attributes,
    'data': df.values.tolist()
}

# Exporta para .arff
output_path = '/data/isa/EEG_Eye_State_com_timestamp.arff'
with open(output_path, 'w') as f:
    arff.dump(arff_data, f)

print(f"Arquivo salvo com sucesso em: {output_path}")

df

       AF3       F7       F3      FC5       T7       P7       O1       O2  \
0  4329.23  4009.23  4289.23  4148.21  4350.26  4586.15  4096.92  4641.03   
1  4324.62  4004.62  4293.85  4148.72  4342.05  4586.67  4097.44  4638.97   
2  4327.69  4006.67  4295.38  4156.41  4336.92  4583.59  4096.92  4630.26   
3  4328.72  4011.79  4296.41  4155.90  4343.59  4582.56  4097.44  4630.77   
4  4326.15  4011.79  4292.31  4151.28  4347.69  4586.67  4095.90  4627.69   

        P8       T8      FC6       F4       F8      AF4 eyeDetection  \
0  4222.05  4238.46  4211.28  4280.51  4635.90  4393.85            0   
1  4210.77  4226.67  4207.69  4279.49  4632.82  4384.10            0   
2  4207.69  4222.05  4206.67  4282.05  4628.72  4389.23            0   
3  4217.44  4235.38  4210.77  4287.69  4632.31  4396.41            0   
4  4210.77  4244.10  4212.82  4288.21  4632.82  4398.46            0   

                   Timestamp  
0 2025-01-01 00:00:00.000000  
1 2025-01-01 00:00:00.007800  
2 2025-01-0

,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4,eyeDetection,Timestamp
0,4329.23,4009.23,4289.23,4148.21,4350.26,4586.15,4096.92,4641.03,4222.05,4238.46,4211.28,4280.51,4635.90,4393.85,0,2025-01-01 00:00:00.000000
1,4324.62,4004.62,4293.85,4148.72,4342.05,4586.67,4097.44,4638.97,4210.77,4226.67,4207.69,4279.49,4632.82,4384.10,0,2025-01-01 00:00:00.007800
2,4327.69,4006.67,4295.38,4156.41,4336.92,4583.59,4096.92,4630.26,4207.69,4222.05,4206.67,4282.05,4628.72,4389.23,0,2025-01-01 00:00:00.015600
3,4328.72,4011.79,4296.41,4155.90,4343.59,4582.56,4097.44,4630.77,4217.44,4235.38,4210.77,4287.69,4632.31,4396.41,0,2025-01-01 00:00:00.023400
4,4326.15,4011.79,4292.31,4151.28,4347.69,4586.67,4095.90,4627.69,4210.77,4244.10,4212.82,4288.21,4632.82,4398.46,0,2025-01-01 00:00:00.031200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11570,4299.49,4018.97,4267.69,4124.10,4344.62,4626.67,4076.41,4612.31,4190.77,4225.64,4202.56,4267.18,4587.69,4368.72,1,2025-01-01 00:01:30.246000
11571,4307.18,4016.92,4274.36,4127.18,4345.64,4624.10,4076.92,4608.72,4188.72,4223.59,4204.62,4275.90,4584.62,4375.90,1,2025-01-01 00:01:30.253800
11572,4305.64,4021.03,4270.26,4128.21,4350.77,4627.18,4074.36,4608.21,4181.54,4218.46,4196.41,4273.33,4578.46,4371.79,1,2025-01-01 00:01:30.261600
11573,4299.49,4024.62,4263.59,4131.79,4353.33,4627.18,4074.87,4611.28,4186.15,4222.05,4193.33,4266.15,4581.03,4364.10,1,2025-01-01 00:01:30.269400


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from copy import deepcopy
from mlex.features.sequences import SequenceTransformer
from mlex import FeatureStratifiedSplit
from mlex import PreProcessingTransformer
from mlex import DataReader
from mlex import StandardEvaluator
from mlex import F1MaxThresholdStrategy
from mlex.models.models import RNNModel

In [ ]:
import arff
import seaborn as sns
import matplotlib.pyplot as plt

data_path = r'/data/isa/EEG_Eye_State.arff'

with open(data_path, 'r') as f:
    raw = arff.load(f)

columns = [attr[0] for attr in raw['attributes']]
df = pd.DataFrame(raw['data'], columns=columns)

df = df.drop([10386,11509, 898]) 


data = df

corr = data.corr(method='spearman')

mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(17, 15))

cmap = sns.diverging_palette(230, 20, as_cmap=True)

heatmap = sns.heatmap(
    corr, 
    mask=mask, 
    cmap=cmap, 
    vmax=0.3, 
    center=0,
    square=True, 
    linewidths=0.5, 
    cbar_kws={"shrink": 0.5},  
    annot=True,  
    fmt=".2f",   
    annot_kws={"size": 8},  
    xticklabels=corr.columns,  
    yticklabels=corr.columns  
)

heatmap.set_xticklabels(heatmap.get_xticklabels(), fontsize=10)
heatmap.set_yticklabels(heatmap.get_yticklabels(), fontsize=10)

plt.tight_layout()
plt.show()

plt.close()

In [ ]:
path = r'/data/isa/EEG_Eye_State.arff'
target_column = 'eyeDetection'
reader = DataReader(path, target_columns=[target_column], filter_dict=[])
X = reader.fit_transform(X=None)
y = reader.get_target()


## linhas de outlier
X = X.drop([10386,11509,898]) 


In [ ]:

preprocessor = PreProcessingTransformer(target_columns=[target_column],numeric_features=X.columns,categorical_features=[], handle_unknown='ignore')
preprocessor.fit(X, y)

X_array = preprocessor.transform(X, y)
y_array = preprocessor.get_target()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.stats import norm
from scipy.stats import norm

class ConfidenceInterval:
  def __init__(self, data, alpha) -> None:
    bootstrap = Bootstrap(data)
    bootstrap.calculate_bootstrap(bootstraps=9000, estimator=np.mean)
    self.mean = bootstrap.mean()
    self.stand = bootstrap.std()
    self.z = norm.ppf(1 - (alpha/2))

  def calculate_lower_bound(self):
    return self.mean - (self.z * self.stand)

  def calculate_upper_bound(self):
    return self.mean + (self.z * self.stand)

class Bootstrap:

    def __init__(self, X):
        self.X = X
        self.bootstraps = []

    def calculate_bootstrap(self, bootstraps=9000, estimator=np.mean):
        for i in range(bootstraps):
            X_sample = np.random.choice(self.X, size=len(self.X), replace=True)
            o = estimator(X_sample)
            self.bootstraps.append(o)

    def mean(self):
        mean = np.sum(self.bootstraps) / len(self.bootstraps)
        return mean

    def std(self):
        mean = self.mean()
        sum_of_diff = 0
        for xi in self.bootstraps:
            sum_of_diff += (xi-mean) * (xi-mean)
        stand = sum_of_diff/ (len(self.bootstraps) - 1)
        stand = np.sqrt(stand)
        return stand
    

K_range = range(1, 51)
n_runs = 30 
alpha = 0.05 # alpha para o intervalo de confiança

all_inertias = np.zeros((len(K_range), n_runs)) 

for i, k in enumerate(K_range):
    for run in range(n_runs):
        kmeans = KMeans(n_clusters=k, n_init="auto", random_state=run)
        kmeans.fit(X_array) 
        all_inertias[i, run] = kmeans.inertia_

means = []
lower_bounds = []
upper_bounds = []

# o bootstrap está sendo calculado dentro da classe confidence interval
for i, k in enumerate(K_range):
    ci = ConfidenceInterval(all_inertias[i, :], alpha)  
    means.append(ci.mean)
    lower_bounds.append(ci.calculate_lower_bound())
    upper_bounds.append(ci.calculate_upper_bound())

single_run_inertias = []
for k in K_range:
    kmeans = KMeans(n_clusters=k, n_init="auto", random_state=0)
    kmeans.fit(X_array) 
    single_run_inertias.append(kmeans.inertia_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 5))

ax1.plot(K_range, means, marker='o', label='Inércia Média')
ax1.fill_between(K_range, lower_bounds, upper_bounds, alpha=0.2, label='Intervalo de Confiança (95%)')
ax1.set_xlabel('Número de clusters (k)')
ax1.set_ylabel('Inércia (Soma dos Erros Quadráticos)')
ax1.set_title('Método do Cotovelo com Intervalo de Confiança')
ax1.grid(True)
ax1.legend()

ax2.plot(K_range, single_run_inertias, marker='o', color='green', label='Inércia (1 Run)')
ax2.set_xlabel('Número de clusters (k)')
ax2.set_ylabel('Inércia (Soma dos Erros Quadráticos)')
ax2.set_title('Método do Cotovelo (Única Execução)')
ax2.grid(True)
ax2.legend()

focus_range = range(1, 16)
ax1.set_xlim(0.5, 15.5)
ax1.set_xticks(focus_range)

ax2.set_xlim(0.5, 15.5)
ax2.set_xticks(focus_range)


plt.tight_layout()
plt.show()
plt.close()

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, n_init="auto", random_state=42)
kmeans.fit(X_array)
X['cluster'] = kmeans.labels_
X


In [ ]:
import pandas as pd

# Caminho original e novo arquivo corrigido
input_path = path
cleaned_path = r'/data/isa/clean_EEG_Eye_State.arff'

# Leitura do arquivo linha a linha
with open(input_path, 'r') as f:
    lines = f.readlines()

# Separar cabeçalho e dados
header_lines = []
data_lines = []
in_data_section = False
expected_num_values = 0

for line in lines:
    stripped = line.strip()
    if not in_data_section:
        header_lines.append(line)
        if stripped.lower().startswith('@attribute'):
            expected_num_values += 1
        elif stripped.lower() == '@data':
            in_data_section = True
    else:
        data_lines.append(line)

# Filtrar linhas válidas
clean_data_lines = []
corrupted_count = 0

for line in data_lines:
    values = line.strip().split(',')
    if len(values) == expected_num_values:
        clean_data_lines.append(line)
    else:
        corrupted_count += 1

# Escreve novo arquivo corrigido
with open(cleaned_path, 'w') as f:
    f.writelines(header_lines)
    f.writelines(clean_data_lines)

print(f"Linhas corrompidas removidas: {corrupted_count}")

import arff

with open(cleaned_path, 'r') as f:
    raw = arff.load(f)

columns = [attr[0] for attr in raw['attributes']]
df = pd.DataFrame(raw['data'], columns=columns)

df.head()

